# 01b — Native-text language detection

**Phase 1b** (plan §6 Stage 1b). Runs immediately after `01_ingestion.ipynb`.

Goal: tag every native-text `PAGE` node with a language verdict before any
downstream stage looks at it. OCR pages (`mode='ocr'`) are deliberately
skipped here — they have no text yet; Phase 3 (post-fusion) will run the
authoritative detector on the fused output and overwrite these properties.

Pipeline:

`(:PAGE {mode:'native_text'}).text -> apps.backend.lang.detect_language(text) -> PAGE.{language, scriptMix, kuntenMarks, langConfidence, langDetectionRule}`

Concretely:

1. **Detector** — `apps.backend.lang.detector.detect_language(text)` runs script-class
   heuristics over Unicode blocks (hiragana / katakana / kanji / kanbun marks /
   latin / digits / CJK punct) and applies six deterministic rules:
   `kunten-marks`, `kana-density`, `latin-mix`, `modern-digit`,
   `simplified-sniff`, `classical-default` (plus `empty` / `too-short` for
   degenerate inputs). Every decision is explainable — the rule name is
   written to `PAGE.langDetectionRule` so Phase 11 (HITL) can audit it.
2. **Orchestrator** — `apps.backend.pipeline.lang_detect.detect_pages(driver)`
   walks `(:PAGE)` with `mode='native_text'` and no `language` set yet, runs
   the detector, and idempotently writes the five language properties via
   batched parameterized Cypher. `recompute_existing=True` reclassifies every
   native page (used after threshold tuning).
3. **Summary** — `language_summary(driver)` rolls up `by_language`,
   `by_tier_language`, and `by_rule` so we can eyeball the corpus distribution.

**Inputs**

- `notebooks/_artifacts/00_setup_smoke_test/health.json` — must report all-healthy.
- `notebooks/_artifacts/01_ingestion/ingestion.json` — produced by 01.
- A live Neo4j with at least one `(:PAGE {mode:'native_text'})` node.

**Outputs**

- `notebooks/_artifacts/01b_language_detection_native/lang_detect.json` —
  per-document totals, by-rule counts, mean confidence, sample decisions.
- Neo4j: every native page now carries `language` + `scriptMix` +
  `kuntenMarks` + `langConfidence` + `langDetectionRule` + `langDetectionAt`.

**Next**: `01c_tier_loader.ipynb` (bulk ingest of `raw/Primary` + `raw/Secondary`).

In [1]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import json
import logging
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'pyproject.toml').exists(), f'cannot locate repo root from {Path.cwd()}'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

loaded = load_dotenv(REPO_ROOT / '.env')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')

ARTIFACT_DIR = REPO_ROOT / 'notebooks' / '_artifacts' / '01b_language_detection_native'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PRIOR_HEALTH    = REPO_ROOT / 'notebooks' / '_artifacts' / '00_setup_smoke_test' / 'health.json'
PRIOR_INGESTION = REPO_ROOT / 'notebooks' / '_artifacts' / '01_ingestion' / 'ingestion.json'

print(f'repo root      : {REPO_ROOT}')
print(f'.env loaded    : {loaded}')
print(f'artifact dir   : {ARTIFACT_DIR}')
print(f'NEO4J_URI      : {os.getenv("NEO4J_URI", "<unset>")}')
print(f'EMBEDDING_DIMS : {os.getenv("EMBEDDING_DIMS", "<unset>")}')

repo root      : /Users/mohasani/Ancient
.env loaded    : True
artifact dir   : /Users/mohasani/Ancient/notebooks/_artifacts/01b_language_detection_native
NEO4J_URI      : bolt://localhost:7687
EMBEDDING_DIMS : 1024


## Refuse to advance if Phase 0 / 1 are red

The Phase 0 health probe must be all-green and the Phase 1 ingest artifact
must exist with at least one document and one PAGE — otherwise this notebook
has nothing to classify.

In [2]:
assert PRIOR_HEALTH.exists(), (
    'run notebooks/00_setup_smoke_test.ipynb first to generate '
    f'{PRIOR_HEALTH}'
)
health = json.loads(PRIOR_HEALTH.read_text(encoding='utf-8'))
for service in ('silra', 'neo4j', 'minio'):
    block = health.get(service, {})
    print(f'  {service:<6}: ok={block.get("ok")} errors={block.get("errors", [])}')
    assert block.get('ok'), f'{service} probe was red — fix it before running 01b'

assert PRIOR_INGESTION.exists(), (
    'run notebooks/01_ingestion.ipynb first to generate '
    f'{PRIOR_INGESTION}'
)
ingestion = json.loads(PRIOR_INGESTION.read_text(encoding='utf-8'))
totals = ingestion.get('neo4j_totals', {})
print(f'\n  ingest totals: documents={totals.get("documents")}, '
      f'pages={totals.get("pages")}, next_links={totals.get("next_links")}')
assert (totals.get('pages') or 0) > 0, (
    'no PAGE nodes in Neo4j — re-run 01_ingestion.ipynb so the detector has data'
)

  silra : ok=True errors=[]
  neo4j : ok=True errors=[]
  minio : ok=True errors=[]

  ingest totals: documents=74, pages=18272, next_links=14626


## 1. Wire up the Neo4j driver

Same client we used in 01 — we re-open it here so the notebook can be re-run
in isolation after a kernel restart.

In [3]:
from apps.backend.graph.neo4j_client import get_driver, ping as neo4j_ping

driver = get_driver()
probe = neo4j_ping(driver)
print(json.dumps({k: v for k, v in probe.items() if k != 'errors'}, indent=2))
assert probe.get('ok'), f'Neo4j ping failed: {probe.get("errors")}'

{
  "ok": true,
  "uri": "bolt://localhost:7687",
  "server_version": "5.18.1",
  "edition": "community",
  "database": "neo4j",
  "constraint_count": 16,
  "vector_index_count": 5
}


## 2. Self-test fixtures (detector × 6 rules)

Before touching Neo4j we exercise `detect_language` on hand-crafted strings
covering every decision path. This is the same set the production module
ships in `SELF_TEST_FIXTURES`; mismatches here mean a threshold drifted or a
Unicode block was mis-classified.

In [4]:
from apps.backend.lang import detect_language
from apps.backend.lang.detector import SELF_TEST_FIXTURES

fixture_results: list[dict] = []
wins = 0
for name, (text, expected) in SELF_TEST_FIXTURES.items():
    profile = detect_language(text)
    ok = profile.language == expected
    if ok:
        wins += 1
    fixture_results.append({
        'fixture': name,
        'expected': expected,
        'actual': profile.language,
        'rule': profile.rule,
        'confidence': profile.confidence,
        'kunten_marks': profile.kunten_marks,
        'char_count': profile.char_count,
    })
    flag = 'OK ' if ok else 'FAIL'
    print(
        f'  {flag}  {name:25s} {expected:14s} -> {profile.language:14s} '
        f'rule={profile.rule:20s} conf={profile.confidence:.2f}'
    )

print(f'\n  {wins}/{len(SELF_TEST_FIXTURES)} fixtures passed')
assert wins == len(SELF_TEST_FIXTURES), (
    f'detector regressed on built-in fixtures — review apps/backend/lang/detector.py thresholds'
)

  OK   tang-classical-prose      zh-classical   -> zh-classical   rule=classical-default    conf=0.95
  OK   modern-vernacular-zh      zh-modern      -> zh-modern      rule=modern-digit         conf=0.85
  OK   kanbun-with-marks         kanbun         -> kanbun         rule=kunten-marks         conf=0.95
  OK   japanese-modern           ja             -> ja             rule=kana-density         conf=0.95
  OK   mixed-academic            mixed          -> mixed          rule=latin-mix            conf=0.85
  OK   empty                     unknown        -> unknown        rule=empty                conf=0.00
  OK   too-short                 unknown        -> unknown        rule=too-short            conf=0.00

  7/7 fixtures passed


## 3. Script-class composition demo

The `scriptMix` field (a JSON-encoded fractional decomposition) is what
downstream stages will read when they need to decide whether a page needs
kanbun-aware tokenization vs Mandarin tokenization vs the bilingual
(`zh+ja`) tokenizer router. Verifies the ratios sum to ~1.0 for every
fixture.

In [5]:
from apps.backend.lang.detector import script_mix

for name, (text, _) in SELF_TEST_FIXTURES.items():
    mix = script_mix(text)
    d = mix.to_dict()
    total = sum(d.values())
    parts = ' '.join(
        f'{k}={v:.2f}' for k, v in d.items() if v >= 0.01
    ) or '(empty)'
    print(f'  {name:25s} sum={total:.2f}  {parts}')
    assert text == '' or abs(total - 1.0) < 0.01, (
        f'{name}: script_mix did not normalize to 1.0 (got {total})'
    )

  tang-classical-prose      sum=1.00  kanji=1.00
  modern-vernacular-zh      sum=1.00  kanji=0.88 digits=0.06 cjk_punct=0.06
  kanbun-with-marks         sum=1.00  kanji=0.62 kanbun_marks=0.25 cjk_punct=0.12
  japanese-modern           sum=1.00  hiragana=0.36 kanji=0.59 cjk_punct=0.05
  mixed-academic            sum=1.00  kanji=0.09 latin=0.64 digits=0.06 cjk_punct=0.07 whitespace=0.14
  empty                     sum=0.00  (empty)
  too-short                 sum=1.00  kanji=1.00


## 4. Run the detector over native-text PAGE nodes

Calls `detect_pages(driver)` with `recompute_existing=True` so the run is
stable on re-execution (without it, a kernel restart that re-runs the cell
would silently no-op once every page has been classified). For production
the orchestrator default is `False` — only newly-ingested pages are
touched.

In [6]:
from apps.backend.pipeline.lang_detect import detect_pages

report = detect_pages(driver, recompute_existing=True, sample_size=8)

print(f'  pages processed : {report.pages_processed}')
print(f'  pages written   : {report.pages_written}')
print(f'  pages skipped   : {report.pages_skipped}')
print(f'  duration        : {report.duration_seconds}s')
print(f'  mean confidence : {report.mean_confidence:.3f}')
if report.errors:
    print(f'  errors          : {report.errors}')
print(f'\n  by language : {dict(sorted(report.by_language.items()))}')
print(f'  by rule     : {dict(sorted(report.by_rule.items()))}')
print(f'  by tier x language:')
for tier, langs in sorted(report.by_tier_language.items()):
    print(f'    {tier:10s} {dict(sorted(langs.items()))}')

assert not report.errors, f'lang_detect.detect_pages reported errors: {report.errors}'
if report.pages_processed > 0:
    assert report.pages_written == report.pages_processed, (
        f'write count mismatch: processed={report.pages_processed} '
        f'written={report.pages_written}'
    )

  pages processed : 11745
  pages written   : 11745
  pages skipped   : 0
  duration        : 110.888s
  mean confidence : 0.898

  by language : {'ja': 1, 'mixed': 27, 'unknown': 167, 'zh-classical': 9979, 'zh-modern': 1571}
  by rule     : {'classical-default': 9979, 'kana-density': 1, 'latin-mix': 27, 'modern-digit': 1386, 'no-rule-matched': 167, 'simplified-sniff': 185}
  by tier x language:
    primary    {'unknown': 104, 'zh-classical': 6224, 'zh-modern': 1}
    secondary  {'ja': 1, 'mixed': 27, 'unknown': 63, 'zh-classical': 3755, 'zh-modern': 1570}


## 5. Sample per-page decisions

Eight pages picked off the head of the run so we can eyeball them. Primary
EPUBs should typically come out as `zh-classical` / `classical-default`;
the modern academic PDF (`田子爽_唐代制举孝悌类科目考论.pdf`) should land in
`zh-modern` (`modern-digit` rule).

In [7]:
for d in report.sample_decisions:
    print(
        f'  {d["language"]:14s} {d["langDetectionRule"]:20s} '
        f'conf={d["langConfidence"]:.2f} kunten={d["kuntenMarks"]!s:<5} '
        f'tier={d["tier"]:9s} pageId={d["pageId"]}'
    )
    mix = d['scriptMix']
    parts = ' '.join(f'{k}={v:.2f}' for k, v in mix.items() if v >= 0.01) or '(empty)'
    print(f'    scriptMix: {parts}')

  zh-classical   classical-default    conf=0.95 kunten=False tier=secondary pageId=刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00003
    scriptMix: kanji=0.82 latin=0.01 digits=0.03 cjk_punct=0.04 whitespace=0.10
  zh-modern      modern-digit         conf=0.85 kunten=False tier=secondary pageId=刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00004
    scriptMix: kanji=0.78 latin=0.03 digits=0.06 cjk_punct=0.03 whitespace=0.09
  zh-classical   classical-default    conf=0.95 kunten=False tier=secondary pageId=刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00005
    scriptMix: kanji=0.88 cjk_punct=0.09 whitespace=0.02
  zh-classical   classical-default    conf=0.95 kunten=False tier=secondary pageId=刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00006
    scriptMix: kanji=0.84 latin=0.02 digits=0.02 cjk_punct=0.08 whitespace=0.03 other=0.01
  zh-classical   classical-default    conf=0.95 kunten=False tier=secondary pageId=刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00007
    scriptMix: kanji=0.83 latin=0.02 digits=0.01 cjk_punct=0.0

## 6. Verify the Neo4j writes round-trip

Three checks:

1. `language_summary(driver)` shows the same totals as the in-memory report.
2. Every native page carries the five language properties (no `(unset)`
   bucket for `mode='native_text'`).
3. `scriptMix` JSON round-trips to a dict whose values sum to ~1.0.

In [8]:
from apps.backend.pipeline.lang_detect import language_summary

summary = language_summary(driver)
print('  by_language:')
for lang, n in sorted(summary['by_language'].items()):
    print(f'    {lang:14s} {n}')
print('\n  by_rule:')
for rule, n in sorted(summary['by_rule'].items()):
    print(f'    {rule:25s} {n}')

with driver.session() as session:
    unset = session.run(
        '''
        MATCH (p:PAGE {mode: 'native_text'})
        WHERE p.language IS NULL
        RETURN count(p) AS n
        '''
    ).single()['n']
    print(f'\n  native PAGEs missing language: {unset}')
    assert unset == 0, (
        f'{unset} native PAGE nodes still lack a language property — detector did not cover them'
    )

    spot = session.run(
        '''
        MATCH (p:PAGE {mode: 'native_text'})
        WHERE p.scriptMix IS NOT NULL
        RETURN p.id AS id, p.language AS language, p.scriptMix AS scriptMix
        LIMIT 3
        '''
    ).data()
    print('\n  scriptMix round-trip:')
    for row in spot:
        mix = json.loads(row['scriptMix'])
        total = sum(mix.values())
        print(f'    {row["language"]:14s} sum={total:.3f}  id={row["id"]}')
        assert abs(total - 1.0) < 0.02, (
            f'scriptMix did not round-trip with sum~=1.0 (got {total}) for {row["id"]}'
        )

  by_language:
    (unset)        6524
    ja             1
    mixed          27
    unknown        167
    zh-classical   9981
    zh-modern      1572

  by_rule:
    (unset)                   6524
    classical-default         9981
    kana-density              1
    latin-mix                 27
    modern-digit              1387
    no-rule-matched           167
    simplified-sniff          185

  native PAGEs missing language: 0

  scriptMix round-trip:
    zh-modern      sum=1.000  id=刘海峰_科举制的起源与进士科的起始__1e5baf7b31::p00000
    zh-modern      sum=1.000  id=刘海峰_科举制的起源与进士科的起始__1e5baf7b31::p00001
    zh-modern      sum=1.000  id=刘海峰_科举制的起源与进士科的起始__1e5baf7b31::p00002


## 7. Persist the run artifact

`lang_detect.json` is the contract for any downstream notebook (01c, 03b,
06, 09) that wants to refuse to run if Phase 1b hasn't classified the
corpus yet.

In [9]:
artifact = {
    'phase': '01b_language_detection_native',
    'ts': datetime.now(timezone.utc).isoformat(),
    'fixture_results': fixture_results,
    'run_report': report.to_dict(),
    'neo4j_summary': summary,
    'inputs': {
        'health': str(PRIOR_HEALTH.relative_to(REPO_ROOT)),
        'ingestion': str(PRIOR_INGESTION.relative_to(REPO_ROOT)),
    },
}
out = ARTIFACT_DIR / 'lang_detect.json'
out.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'wrote {out} ({out.stat().st_size} bytes)')
print('next: 01c_tier_loader.ipynb')

print('01b done. driver intentionally left open — re-run the `clients` cell if you closed it by hand.')

wrote /Users/mohasani/Ancient/notebooks/_artifacts/01b_language_detection_native/lang_detect.json (9052 bytes)
next: 01c_tier_loader.ipynb
01b done. driver intentionally left open — re-run the `clients` cell if you closed it by hand.
